# 2. Jesuit authors: one list from three sources

1. **Wikidata:** I take the people whose religious order is the Society of Jesus (Q36380), born before 1790 or with no birth date.
2. **Mission library inventories:** I take the authors with affiliation "Jesuita" in my dataset of Jesuit mission libraries
   in South America (jesuit-missionary-libraries).
3. **BNE authority file (persons):** I take the records related to the Compañía de Jesús (group, occupation or attributes),
   born before 1790 or with no birth date.

I compare names as sets of words: "Juan Eusebio Nieremberg" = "Nieremberg, Juan Eusebio, (1595-1658)".

---

# 2. Autores jesuitas: una lista a partir de tres fuentes

1. **Wikidata:** tomo las personas cuya orden religiosa es la Compañía de Jesús (Q36380), nacidas antes de 1790 o sin fecha de
   nacimiento.
2. **Inventarios de bibliotecas misionales:** tomo los autores con afiliación "Jesuita" de mi base de bibliotecas de las misiones
   jesuitas de América del Sur (jesuit-missionary-libraries).
3. **Archivo de autoridades de la BNE (personas):** tomo los registros vinculados con la Compañía de Jesús (grupo, ocupación u
   otros atributos), nacidos antes de 1790 o sin fecha de nacimiento.

Comparo los nombres como conjuntos de palabras: "Juan Eusebio Nieremberg" = "Nieremberg, Juan Eusebio, (1595-1658)".

In [1]:
# Celda 1
# Paths and parameters:
%cd "E:/No sincroniza/Software desarrollado/spanish-imprints-counts"
DATA = "E:/Datos/spanish_imprints"
WIKIDATA = DATA + "/wikidata_jesuits.json"
LIBRARY_AUTHORS = "C:/Users/fabi_/Documents/workflow/projects/jesuit-missionary-libraries/datos/authors.csv"
LIBRARY_INSTITUTIONS = "C:/Users/fabi_/Documents/workflow/projects/jesuit-missionary-libraries/datos/institutions.csv"
BNE_PERSONS = DATA + "/bne/persona-UTF8.zip"     # Downloaded by hand from datosabiertos.bne.es (1.2 GB unzipped; read in chunks).
OUT = DATA + "/jesuit_authors.tsv"
BORN_BEFORE = 1790

E:\No sincroniza\Software desarrollado\spanish-imprints-counts


In [2]:
# Celda 2
import os, json, subprocess, zipfile
import pandas as pd
from helpers import name_key

## 1. Which Jesuits does Wikidata list? / ¿Qué jesuitas registra Wikidata?

In [3]:
# Celda 3
# Downloads once (about 7 MB) and keeps the file:
if not os.path.exists(WIKIDATA):
    query = f"""SELECT ?p ?name WHERE {{ ?p wdt:P611 wd:Q36380 .
      OPTIONAL {{ ?p wdt:P569 ?b }} FILTER(!BOUND(?b) || YEAR(?b) < {BORN_BEFORE})
      ?p rdfs:label|skos:altLabel ?name . FILTER(LANG(?name) IN ("es", "la", "en", "it", "pt", "fr", "ca")) }}"""
    out = subprocess.run(["curl", "-s", "-m", "300", "-G", "-H", "Accept: application/sparql-results+json",
                          "-A", "spanish-imprints-counts/0.1 (research)", "--data-urlencode", "query=" + query,
                          "https://query.wikidata.org/sparql"], capture_output=True).stdout
    open(WIKIDATA, "wb").write(out)

rows = json.load(open(WIKIDATA, encoding="utf8"))["results"]["bindings"]
wikidata = pd.DataFrame({"person": [r["p"]["value"] for r in rows], "name": [r["name"]["value"] for r in rows]})
wikidata["source"] = "wikidata"
print("People:", wikidata["person"].nunique(), "| name forms:", len(wikidata))
wikidata.head()

People: 4863 | name forms: 28314


,person,name,source
0,http://www.wikidata.org/entity/Q282996,Ferenc Abel,wikidata
1,http://www.wikidata.org/entity/Q283121,Josef Dobrovsky,wikidata
2,http://www.wikidata.org/entity/Q283121,Joseph Dobrovski,wikidata
3,http://www.wikidata.org/entity/Q283121,Joseph Dobrovský,wikidata
4,http://www.wikidata.org/entity/Q283121,Abad Azul,wikidata


## 2. Which Jesuit authors appear in the mission library inventories? / ¿Qué autores jesuitas aparecen en los inventarios de las bibliotecas misionales?

In [4]:
# Celda 4
authors = pd.read_csv(LIBRARY_AUTHORS)
institutions = pd.read_csv(LIBRARY_INSTITUTIONS)
authors = authors.merge(institutions, on="institution", how="left")
print(authors["affiliation"].value_counts())

library = authors[authors["affiliation"] == "Jesuita"][["author"]].rename(columns={"author": "name"})
library["person"] = "library:" + library["name"]
library["source"] = "library"
print("Jesuit authors:", len(library))

Jesuita                          316
Laico católico                   122
Otra orden religiosa católica     95
Clero secular                     76
Franciscano                       44
-                                 26
Pagano                            26
Dominico                          23
Otro                              19
Name: affiliation, dtype: int64
Jesuit authors: 316


## 3. Which persons does the BNE authority file link to the Society of Jesus? / ¿Qué personas vincula con la Compañía de Jesús el archivo de autoridades de la BNE?

In [5]:
# Celda 5
# Reads the 1.2 GB file in chunks and keeps only the rows related to the Society of Jesus (name, group, occupation, attributes):
columns = ["id", "nombre_de_persona", "otros_nombres", "fecha_nacimiento", "grupo_o_entidad_relacionada",
           "ocupacion", "otros_atributos_persona"]
pattern = r"Compañía de Jesús|Jesuitas|Jesuita|\(S\.\s?I\.\)"  # "(S.I.)" after the name.
parts = []
with zipfile.ZipFile(BNE_PERSONS).open("per/persona-UTF8.csv") as f:
    for chunk in pd.read_csv(f, sep=";", dtype=str, usecols=columns, chunksize=200_000):
        text = chunk["nombre_de_persona"].fillna("") + " " + chunk["grupo_o_entidad_relacionada"].fillna("") + " " + chunk["ocupacion"].fillna("") + " " + chunk["otros_atributos_persona"].fillna("")
        parts.append(chunk[text.str.contains(pattern)])
bne = pd.concat(parts)
print("BNE persons related to the Society of Jesus:", len(bne))
bne[["nombre_de_persona", "fecha_nacimiento", "grupo_o_entidad_relacionada", "ocupacion"]].head(10)

BNE persons related to the Society of Jesus: 3224


,nombre_de_persona,fecha_nacimiento,grupo_o_entidad_relacionada,ocupacion
3101,"García, Diego, ( 1634-1716)",1634,Jesuitas,"Sacerdotes, Profesores universitarios, Teólogos"
4924,"Le Camus, André, ( 1663-1740)",1663,Jesuitas,Religiosos
5812,"Álvarez, Gabriel, ( 1564-1645)",1564,Jesuitas,"Religiosos, Historiadores, Profesores"
13607,"Miranda, Francisco, ( 1720-1787)",1720,Jesuitas,"Religiosos, Lexicógrafos"
15218,"Álvarez, Gabriel, ( 1641-1702)",1641,Jesuitas,Religiosos
17420,"Dez, Jean, (S.I.)",NaN,NaN,NaN
17421,"Dez, Jean, (S.I.)",NaN,NaN,NaN
18324,"López, Antonio, ( 1857-1903)",1857,Jesuitas,"Profesores, Religiosos, embne"
19959,"Lobo, Jerónimo, (S.I.), ( 1594-1678)",NaN,NaN,NaN
19965,"Lobo, Jerónimo, (S.I.), ( 1594-1678)",NaN,NaN,NaN


In [6]:
# Celda 6
# Birth year: "1595" -> 1595, "ca. 1600" -> 1600; modern Jesuits are left out:
bne["born"] = pd.to_numeric(bne["fecha_nacimiento"].str.extract(r"(\d{4})")[0], errors="coerce")
print(bne["born"].describe())
bne = bne[bne["born"].isna() | (bne["born"] < BORN_BEFORE)]
print("Born before", BORN_BEFORE, "or unknown:", len(bne))

count     964.000000
mean     1795.056017
std       148.636436
min      1500.000000
25%      1641.000000
50%      1875.500000
75%      1923.250000
max      1968.000000
Name: born, dtype: float64
Born before 1790 or unknown: 2637


In [7]:
# Celda 7
# Main name plus variant names ("Nieremberg, Juan Eusebio /**/ Nieremberg, Ioannes Eusebius"):
names = bne["nombre_de_persona"].fillna("") + " /**/ " + bne["otros_nombres"].fillna("")
bne_names = pd.DataFrame({"person": "bne:" + bne["id"], "name": names}).assign(name=lambda d: d["name"].str.split(r"/\*\*/")).explode("name")
bne_names["name"] = bne_names["name"].str.strip()
bne_names = bne_names[bne_names["name"] != ""]
bne_names["source"] = "bne"
print("Name forms:", len(bne_names))

Name forms: 3949


## 4. How many Jesuit authors do the three sources give together? / ¿Cuántos autores jesuitas dan las tres fuentes juntas?

In [8]:
# Celda 8
# Names with fewer than two words ("Ripalda") are dropped: too ambiguous:
jesuits = pd.concat([wikidata, library, bne_names])
jesuits["key"] = jesuits["name"].map(lambda s: " ".join(sorted(name_key(s))))
jesuits = jesuits[jesuits["key"].str.count(" ") >= 1].drop_duplicates(["key", "source"])
jesuits.to_csv(OUT, sep="\t", index=False)

sources = jesuits.groupby("key")["source"].apply(lambda s: "+".join(sorted(set(s))))
print("Distinct name keys:", len(sources))
print(sources.value_counts())

Distinct name keys: 11490
wikidata                8716
bne                     1963
bne+wikidata             497
library+wikidata         138
bne+library+wikidata      86
library                   68
bne+library               22
Name: source, dtype: int64


**Reading.** The three sources overlap only in part: each one adds names that the others miss. The inventories bring Jesuits
who circulated in mission libraries but have no Wikidata entry.

---

**Lectura.** Las tres fuentes se superponen solo en parte: cada una suma nombres que las otras no tienen. Los inventarios aportan
jesuitas que circularon en las bibliotecas misionales pero no tienen entrada en Wikidata.